# Learning 03: Fine-Tuning GLiNER

GLiNER can be fine-tuned on domain-specific data to improve entity recall in specialized
vocabularies or with custom entity types not well covered by zero-shot inference.

## Training Data Format

```python
{
    "tokenized_text": ["Bill", "Gates", "founded", "Microsoft", "in", "1975"],
    "ner": [[0, 1, "person"], [3, 3, "organization"], [5, 5, "date"]]
}
```

Key points:
- `tokenized_text`: word-level token list (NOT subword tokens)
- `ner`: list of `[start_token, end_token, label]` — **inclusive** token indices
- `[0, 1, "person"]` → tokens 0 and 1 = `"Bill Gates"`
- `[3, 3, "organization"]` → token 3 = `"Microsoft"`

For bi-encoder models, optionally add:
- `ner_labels`: explicit positive labels (controls what the model sees as candidates)
- `ner_negatives`: hard negative labels to improve discrimination

In [1]:
# pip install gliner[training]
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "training_examples.json")) as f:
    train_data = json.load(f)

print(f"✓ Loaded {len(train_data)} training examples")
print(f"\nSample example:")
print(json.dumps(train_data[0], indent=2))

✓ Loaded 20 training examples

Sample example:
{
  "tokenized_text": [
    "Elon",
    "Musk",
    "founded",
    "Tesla",
    "in",
    "2003",
    "in",
    "San",
    "Carlos",
    "California"
  ],
  "ner": [
    [
      0,
      1,
      "person"
    ],
    [
      3,
      3,
      "organization"
    ],
    [
      5,
      5,
      "date"
    ],
    [
      7,
      9,
      "location"
    ]
  ]
}


## 1. Validate Training Data Format

Before training, verify that your data is correctly formatted.

In [2]:
def validate_training_data(examples):
    """Check that all examples have valid GLiNER training format."""
    errors = []
    for i, ex in enumerate(examples):
        tokens = ex["tokenized_text"]
        n = len(tokens)
        for span in ex["ner"]:
            start, end, label = span
            if start < 0 or end >= n:
                errors.append(f"Example {i}: span [{start},{end}] out of bounds (len={n})")
            if start > end:
                errors.append(f"Example {i}: span start {start} > end {end}")
            if not isinstance(label, str) or not label:
                errors.append(f"Example {i}: invalid label {label!r}")
    return errors


errors = validate_training_data(train_data)
if errors:
    for e in errors:
        print(f"✗ {e}")
else:
    print(f"✓ All {len(train_data)} examples are valid")

# Show entity distribution
from collections import Counter
label_counts = Counter(span[2] for ex in train_data for span in ex["ner"])
print(f"\nLabel distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:20} {count}")

✓ All 20 examples are valid

Label distribution:
  person               22
  organization         19
  date                 19
  location             13
  product              1
  event                1


## 2. Converting Raw Text → GLiNER Format

Real-world annotation tools often produce character-level spans.
Here's how to convert them to GLiNER's token-level format.

In [3]:
import re

def char_to_token_offsets(text, char_start, char_end):
    """
    Convert character-level span to token (word) indices.
    Uses simple whitespace tokenization — same approach as GLiNER training.
    Returns (tokens, start_token_idx, end_token_idx) or (tokens, None, None) if not found.
    """
    # Find word boundaries
    tokens = []
    token_spans = []  # (char_start, char_end) for each token
    for m in re.finditer(r'\S+', text):
        tokens.append(m.group())
        token_spans.append((m.start(), m.end()))

    # Find which tokens are within the entity span
    entity_tokens = [
        i for i, (ts, te) in enumerate(token_spans)
        if ts >= char_start and te <= char_end
    ]
    if not entity_tokens:
        return tokens, None, None
    return tokens, entity_tokens[0], entity_tokens[-1]


def from_char_annotations(text, char_entities):
    """
    Convert a text with character-level entity spans to GLiNER training format.

    Args:
        text: raw text string
        char_entities: list of {"text": str, "label": str, "start": int, "end": int}

    Returns:
        {"tokenized_text": [...], "ner": [[start, end, label], ...]}
    """
    ner = []
    tokens = None
    for ent in char_entities:
        toks, start_tok, end_tok = char_to_token_offsets(text, ent["start"], ent["end"])
        if tokens is None:
            tokens = toks
        if start_tok is not None:
            ner.append([start_tok, end_tok, ent["label"]])

    return {"tokenized_text": tokens or text.split(), "ner": ner}


# Example: predict_entities gives char offsets → convert to training format
example_text = "Elon Musk founded SpaceX in Hawthorne California"
# Simulate what predict_entities returns (char-level)
char_ents = [
    {"text": "Elon Musk", "label": "person", "start": 0, "end": 9},
    {"text": "SpaceX", "label": "organization", "start": 18, "end": 24},
    {"text": "Hawthorne California", "label": "location", "start": 28, "end": 48},
]
training_example = from_char_annotations(example_text, char_ents)
print("Converted training example:")
print(json.dumps(training_example, indent=2))

# Verify: reconstruct entity texts from token indices
tokens = training_example["tokenized_text"]
for start, end, label in training_example["ner"]:
    entity_text = " ".join(tokens[start:end+1])
    print(f"  [{label}] tokens[{start}:{end}] = {entity_text!r}")

Converted training example:
{
  "tokenized_text": [
    "Elon",
    "Musk",
    "founded",
    "SpaceX",
    "in",
    "Hawthorne",
    "California"
  ],
  "ner": [
    [
      0,
      1,
      "person"
    ],
    [
      3,
      3,
      "organization"
    ],
    [
      5,
      6,
      "location"
    ]
  ]
}
  [person] tokens[0:1] = 'Elon Musk'
  [organization] tokens[3:3] = 'SpaceX'
  [location] tokens[5:6] = 'Hawthorne California'


## 3. Train the Model

`model.train_model()` wraps HuggingFace Trainer and handles all the GLiNER-specific
data collation and loss computation.

In [4]:
import os
output_dir = os.path.join(os.getcwd(), "..", "trained_model")

model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
print("✓ Base model loaded")

# Split 16 train / 4 eval
train_split = train_data[:16]
eval_split  = train_data[16:]

trainer = model.train_model(
    train_split,
    eval_split,
    output_dir=output_dir,
    max_steps=200,                       # Small for demo; use 2000+ for real fine-tuning
    learning_rate=1e-5,                  # Lower LR for encoder
    others_lr=3e-5,                      # Higher LR for non-encoder params
    per_device_train_batch_size=4,
    negatives=1.5,                       # Higher negative sampling for bi-encoders
    warmup_ratio=0.1,
    save_steps=200,
    save_total_limit=1,
)
trainer.model.save_pretrained(output_dir)
print(f"✓ Training complete, model saved to {output_dir}")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:447: UserWarning: Resizing embeddings is not supported for bi-encoder models.
  instance.resize_embeddings()


✓ Base model loaded


/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:1141: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(**trainer_kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50283}.


Step,Training Loss
10,15.210200
20,4.388500
30,0.094600
40,0.000500
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


✓ Training complete, model saved to /home/biomike/BioMike/nlp-puzzles/modules/15_ner_gliner/learning/../trained_model


## 4. Evaluate Before and After Fine-Tuning

In [5]:
# Load fine-tuned model
finetuned_model = GLiNER.from_pretrained(output_dir)
base_model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")

# Test on a held-out example
test_text = "Howie Liu founded Airtable in San Francisco in 2012"
labels = ["person", "organization", "location", "date"]

base_ents = base_model.predict_entities(test_text, labels, threshold=0.3)
ft_ents = finetuned_model.predict_entities(test_text, labels, threshold=0.3)

print(f"Test: {test_text}\n")
print("Base model:")
for e in base_ents:
    print(f"  [{e['label']:12}] {e['text']!r:25} score={e['score']:.2f}")

print("\nFine-tuned model:")
for e in ft_ents:
    print(f"  [{e['label']:12}] {e['text']!r:25} score={e['score']:.2f}")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:447: UserWarning: Resizing embeddings is not supported for bi-encoder models.
  instance.resize_embeddings()


/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Test: Howie Liu founded Airtable in San Francisco in 2012

Base model:
  [person      ] 'Howie Liu'               score=0.75
  [organization] 'Airtable'                score=0.71
  [location    ] 'San Francisco'           score=0.62
  [date        ] '2012'                    score=0.53

Fine-tuned model:
  [person      ] 'Howie Liu'               score=1.00
  [organization] 'Airtable'                score=1.00
  [location    ] 'San Francisco'           score=1.00
  [date        ] '2012'                    score=1.00


## 5. Adding Explicit Labels and Hard Negatives

For bi-encoder models, providing `ner_labels` and `ner_negatives` per example
improves training by controlling what the label encoder sees during training.

In [6]:
# Enrich training examples with explicit labels and hard negatives
def enrich_example(example, all_labels, n_negatives=3):
    """Add ner_labels and ner_negatives fields to a training example."""
    positive_labels = list({span[2] for span in example["ner"]})
    negative_labels = [l for l in all_labels if l not in positive_labels]
    return {
        **example,
        "ner_labels": positive_labels + negative_labels[:n_negatives],
        "ner_negatives": negative_labels[n_negatives:n_negatives*2],
    }


all_labels = ["person", "organization", "location", "date", "product", "event"]
enriched_data = [enrich_example(ex, all_labels) for ex in train_data]

print("Enriched example:")
print(json.dumps(enriched_data[0], indent=2))

Enriched example:
{
  "tokenized_text": [
    "Elon",
    "Musk",
    "founded",
    "Tesla",
    "in",
    "2003",
    "in",
    "San",
    "Carlos",
    "California"
  ],
  "ner": [
    [
      0,
      1,
      "person"
    ],
    [
      3,
      3,
      "organization"
    ],
    [
      5,
      5,
      "date"
    ],
    [
      7,
      9,
      "location"
    ]
  ],
  "ner_labels": [
    "date",
    "person",
    "organization",
    "location",
    "product",
    "event"
  ],
  "ner_negatives": []
}


## Summary

- GLiNER training format: `{"tokenized_text": [...], "ner": [[start, end, label], ...]}`
- Token indices are **word-level** (not subword) and **inclusive**
- Use `from_char_annotations()` to convert character-level annotations
- `model.train_model()` wraps HuggingFace Trainer — minimal boilerplate
- Bi-encoders: use `negatives=1.5`, lower LR for encoder (`1e-5`), higher for others (`3e-5`)
- Add `ner_labels` + `ner_negatives` per example for better label discrimination
- Even 100-200 steps on 20 examples noticeably shifts confidence scores for in-domain entities